In [11]:
import pandas as pd

# Load the Excel file and inspect sheet names
# file_name = 'Answer only prod.xlsx'
# file_name = 'Paragraph CoT prod.xlsx'
# file_name = 'Step-by-step CoT -- All at once prod.xlsx'
file_name = 'Step-by-step CoT -- Sequential prod.xlsx'
xlsx = pd.ExcelFile(file_name)
print("Available sheets:", xlsx.sheet_names)

# Select the 'main study' sheet (or the first matching name)
sheet = next((s for s in xlsx.sheet_names if 'main study' in s.lower()), xlsx.sheet_names[0])
print("Using sheet:", sheet)

# Read the data
df = pd.read_excel(xlsx, sheet_name=sheet)

# Compute correctness according to the rules:
# 1) If Model Answer == Gt Answer, then 'Accept' is correct.
# 2) If Model Answer != Gt Answer, then 'Reject' is correct.
df['correct'] = ((df['Question idx'] != 1) &  
                 (df['Question idx'] != 2)  & 
                 (df['Question idx'] != 3) &  
                 (df['Question idx'] != 4) &
    (((df['Model Answer'] == df['Gt Answer']) & (df['Step 2'] == 'Accept')) | ((df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Reject')))
)

df['overreliance'] = ((df['Question idx'] != 1) & (df['Question idx'] != 2)  & (df['Question idx'] != 3) & (df['Question idx'] != 4) &
                    (df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Accept'))

# Summary statistics
total = len(df) - df['Question idx'].isin([1, 2, 3, 4]).sum()
correct_count = df['correct'].sum()
accuracy = correct_count / total * 100
overreliance_count = df['overreliance'].sum()

print(file_name)
print(f"Total test questions: {total}")
print(f"Correct: {correct_count}")
print(f"Overreliance: {overreliance_count}")
print(f"Overreliance rate: {overreliance_count / total * 100:.2f}%")
print(f"Accuracy: {accuracy:.2f}%")

# Show the first few rows with the new 'correct' column
# from ace_tools import display_dataframe_to_user
# display_dataframe_to_user("Preview with Correctness", df.head())


Available sheets: ['Demographics', 'Main Study', 'Task Demand Questions', 'AI Usage Questions', 'Free Form Questions', 'Interaction Questions', 'Evaluation', '67c9ee2618920406b694ea35', '67d4405b91b28eb4a6084f22', '673c66f7d2a07d0ccfbeb0a5']
Using sheet: Main Study
Step-by-step CoT -- Sequential prod.xlsx
Total test questions: 36
Correct: 26
Overreliance: 8
Overreliance rate: 22.22%
Accuracy: 72.22%


In [7]:
import pandas as pd

def calculate_accuracy_by_user(
    filename: str,
    sheet_name: str = None
) -> pd.DataFrame:
    # 1. Load workbook & pick the “Main Study” sheet
    xlsx = pd.ExcelFile(filename)
    if sheet_name is None:
        # pick the first sheet whose name contains “main study” (case‐insensitive)
        sheet_name = next(
            s for s in xlsx.sheet_names
            if "main study" in s.lower()
        )
    df = pd.read_excel(xlsx, sheet_name=sheet_name)
    df = (
        df.groupby("Username", group_keys=False)
          .apply(lambda g: g.iloc[4:])   # remove first 4 rows in each group
    )

    # 2. Apply correctness rules
    df['correct'] = ((df['Question idx'] != 1) &  
                 (df['Question idx'] != 2)  & 
                 (df['Question idx'] != 3) &  
                 (df['Question idx'] != 4) &
    (((df['Model Answer'] == df['Gt Answer']) & (df['Step 2'] == 'Accept')) | ((df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Reject'))))
    
    df['overreliance'] = ((df['Question idx'] != 1) & (df['Question idx'] != 2)  & (df['Question idx'] != 3) & (df['Question idx'] != 4) &
                    (df['Model Answer'] != df['Gt Answer']) & (df['Step 2'] == 'Accept'))

    # 3. Group by Username and compute aggregates
    summary = (
        df.groupby('Username')
          .agg(
              total=('correct', 'size'),
              correct=('correct', 'sum'),
              overreliance = ('overreliance', 'sum')
          )
          .reset_index()
    )
    summary['accuracy (%)'] = summary['correct'] / (summary['total']) * 100
    summary['overreliance (%)'] = summary['overreliance'] / (summary['total']) * 100

    return summary

if __name__ == "__main__":
    # point to your file here
    # FPATH = "Answer only prod.xlsx"
    # FPATH = 'Paragraph CoT prod.xlsx'
    # FPATH = 'Step-by-step CoT -- All at once prod.xlsx'
    FPATH = 'Step-by-step CoT -- Sequential prod.xlsx'

    result = calculate_accuracy_by_user(FPATH)
    print(FPATH)
    print(result)

    # optional: save to Excel
    result.to_excel(f'{FPATH}_accuracy_by_user.xlsx', index=False)

Step-by-step CoT -- Sequential prod.xlsx
                   Username  total  correct  overreliance  accuracy (%)  \
0  673c66f7d2a07d0ccfbeb0a5     12        7             4     58.333333   
1  67c9ee2618920406b694ea35     12        9             3     75.000000   
2  67d4405b91b28eb4a6084f22     12       10             1     83.333333   

   overreliance (%)  
0         33.333333  
1         25.000000  
2          8.333333  


/tmp/ipykernel_1410526/2774631722.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[4:])   # remove first 4 rows in each group


Compute the pilot effect size, $w$
1. Observed proportions
$$p_i = \frac{\text{correct}_i}{n_i} \quad\Rightarrow\quad [0.8958,\;0.6389,\;0.7500,\;0.7222] $$

2. Overall proportion under $H₀$ (pooled)
$$p_0 = \frac{\sum_i \text{correct}_i}{\sum_i n_i} = \frac{128}{168} \approx 0.7619$$

3. Cohen’s $w$ for $k$ groups
$$w \;=\;\sqrt{\sum_{i=1}^k \frac{(p_i - p_0)^2}{p_0}}\;\approx\; 0.214$$

In [2]:
# # power analysis
# import statsmodels.stats.api as sms
# def power_analysis(effect_size: float, alpha: float = 0.05, power: float = 0.8) -> int:
#     """
#     Perform a power analysis to determine the required sample size.
    
#     Parameters:
#     - effect_size: The expected effect size (Cohen's d).
#     - alpha: The significance level (default is 0.05).
#     - power: The desired power of the test (default is 0.8).
    
#     Returns:
#     - Required sample size for each group.
#     """
#     return sms.NormalIndPower().solve_power(effect_size=effect_size, alpha=alpha, power=power)
# if __name__ == "__main__":
#     # Example usage
#     effect_size = 0.5  # Example effect size (Cohen's d)
#     required_sample_size = power_analysis(effect_size)
#     print(f"Required sample size for each group with effect size {effect_size}: {required_sample_size:.2f}")
from statsmodels.stats.power import GofChisquarePower

analysis = GofChisquarePower()
w        = 0.214   # from step 2
alpha    = 0.05
power    = 0.80
n_bins   = 4      # number of groups

n_total = analysis.solve_power(effect_size=w,
                               nobs=None,
                               alpha=alpha,
                               power=power,
                               n_bins=n_bins)
print(f"Total N needed: {n_total:.0f}")
# --> total sample size across all conditions

Total N needed: 238
